In [1]:
# import pandas as pd
# import torch
# from transformers import T5Tokenizer, T5ForConditionalGeneration

# # ======================
# # 1. Load Fine-tuned Model
# # ======================
# model_path = "/kaggle/input/gpt2-trained-variants/pytorch/default/8/Important model flan-t5 (first model)/hp_t5_finetuned"   # use the path where you saved after training
# tokenizer = T5Tokenizer.from_pretrained(model_path)
# model = T5ForConditionalGeneration.from_pretrained(model_path)

# if torch.cuda.is_available():
#     model = model.cuda()

# # ======================
# # 2. Load CSV with captions
# # ======================
# csv_path = "/kaggle/input/hp-all/hp_all_cap.csv"  # 👈 change path
# df = pd.read_csv(csv_path)

# if "caption" not in df.columns:
#     raise ValueError(f"'caption' column not found in CSV. Found: {list(df.columns)}")

# captions = df["caption"].fillna("").tolist()

# # ======================
# # 3. Run Inference
# # ======================
# styled_outputs = []

# for cap in captions:
#     if cap.strip() == "":
#         styled_outputs.append("")
#         continue

#     inputs = tokenizer(cap, return_tensors="pt", truncation=True, max_length=128)
#     if torch.cuda.is_available():
#         inputs = {k: v.cuda() for k, v in inputs.items()}

#     with torch.no_grad():
#         output = model.generate(
#             **inputs,
#             max_length=100,
#             num_beams=4,        # beam search for better quality
#             early_stopping=True
#         )

#     styled_text = tokenizer.decode(output[0], skip_special_tokens=True)
#     styled_outputs.append(styled_text)

# # ======================
# # 4. Save Results
# # ======================
# df["styled_caption"] = styled_outputs
# output_path = "captions_with_style.csv"
# df.to_csv(output_path, index=False)

# print(f"✅ Inference complete. Saved to {output_path}")

# # ======================
# # 5. Show Sample
# # ======================
# print("\nSample Results:")
# print("="*80)
# for i in range(min(5, len(df))):
#     print("Original:", df["caption"].iloc[i])
#     print("Styled  :", df["styled_caption"].iloc[i])
#     print("-"*40)


In [2]:
import pandas as pd
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration

# ======================
# 1. Load Fine-tuned Model
# ======================
model_path = "/kaggle/input/gpt2-trained-variants/pytorch/default/9/Important model flan-t5 (second model)/hp_t5_finetuned"   # 👈 use the path where you saved after training
tokenizer = T5Tokenizer.from_pretrained(model_path)
model = T5ForConditionalGeneration.from_pretrained(model_path)

if torch.cuda.is_available():
    model = model.cuda()

# ======================
# 2. Load CSV with captions
# ======================
csv_path = "/kaggle/input/hp-all/hp_all_cap.csv"  # 👈 change path
df = pd.read_csv(csv_path)

if "caption" not in df.columns:
    raise ValueError(f"'caption' column not found in CSV. Found: {list(df.columns)}")

captions = df["caption"].fillna("").tolist()

# ======================
# 3. Run Inference (with prefix)
# ======================
styled_outputs = []

for cap in captions:
    if cap.strip() == "":
        styled_outputs.append("")
        continue

    # 👇 Add the style prefix
    inputs = tokenizer("hp_style: " + cap, return_tensors="pt", truncation=True, max_length=128)
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=100,
            num_beams=4,        # beam search for better quality
            early_stopping=True
        )

    styled_text = tokenizer.decode(output[0], skip_special_tokens=True)
    styled_outputs.append(styled_text)

# ======================
# 4. Save Results
# ======================
df["styled_caption"] = styled_outputs
output_path = "captions_with_style.csv"
df.to_csv(output_path, index=False)

print(f"✅ Inference complete. Saved to {output_path}")

# ======================
# 5. Show Sample
# ======================
print("\nSample Results:")
print("="*80)
for i in range(min(5, len(df))):
    print("Original:", df["caption"].iloc[i])
    print("Styled  :", df["styled_caption"].iloc[i])
    print("-"*40)


2025-09-26 11:53:31.474950: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758887611.705416      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758887611.771093      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


✅ Inference complete. Saved to captions_with_style.csv

Sample Results:
Original: Owl sitting on pivet drive sign board
Styled  : There was a large brown owl sitting on the ground.
----------------------------------------
Original: Professor Dumbledore under street light
Styled  : Dumbledore looked as though he knew what Harry was thinking.
----------------------------------------
Original: Professor Dumbledore and Professor Mcgonagall at door with baby
Styled  : Dumbledore and Professor McGonagall were at the door.
----------------------------------------
Original: Baby with lightning shaped scar
Styled  : The scar on Harry's forehead was prickling.
----------------------------------------
Original: Harry peeking though trapdoor
Styled  : Harry knocked and pushed the door open.
----------------------------------------
